In [3]:
import pandas as pd

df = pd.read_csv("../data/online_retail_II.csv")

print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [4]:
df.isnull().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [5]:
df = df.dropna(subset=["Customer ID"])

In [6]:
df = df[~df["Invoice"].astype(str).str.startswith("C")]

In [7]:
df = df[(df["Quantity"] > 0) & (df["Price"] > 0)]

In [8]:
df["Customer ID"] = df["Customer ID"].astype(int)
df["InvoiceDate"] = pd.to_datetime(df["InvoiceDate"])

In [9]:
df["Revenue"] = df["Quantity"] * df["Price"]

In [10]:
df = df.drop_duplicates()

In [11]:
print(df.shape)
print(df.isnull().sum())
df.describe()

(779425, 9)
Invoice        0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
Price          0
Customer ID    0
Country        0
Revenue        0
dtype: int64


,Quantity,InvoiceDate,Price,Customer ID,Revenue
count,779425.000000,779425,779425.000000,779425.000000,779425.000000
mean,13.489370,2011-01-03 01:44:42.593476,3.218488,15320.360461,22.291823
min,1.000000,2009-12-01 07:45:00,0.001000,12346.000000,0.001000
25%,2.000000,2010-07-02 14:39:00,1.250000,13971.000000,4.950000
50%,6.000000,2010-12-02 14:09:00,1.950000,15247.000000,12.480000
75%,12.000000,2011-08-01 13:44:00,3.750000,16794.000000,19.800000
max,80995.000000,2011-12-09 12:50:00,10953.500000,18287.000000,168469.600000
std,145.855814,NaN,29.676140,1695.692775,227.427075


In [12]:
df.to_csv("../data/cleaned_retail.csv", index=False)

In [15]:
snapshot_date = df["InvoiceDate"].max() + pd.Timedelta(days=1)
print(snapshot_date)

2011-12-10 12:50:00


In [16]:
rfm = df.groupby("Customer ID").agg(
    Recency=("InvoiceDate", lambda x: (snapshot_date - x.max()).days),
    Frequency=("Invoice", "nunique"),
    Monetary=("Revenue", "sum")
).reset_index()

rfm.head()

,Customer ID,Recency,Frequency,Monetary
0,12346,326,12,77556.46
1,12347,2,8,4921.53
2,12348,75,5,2019.40
3,12349,19,4,4428.69
4,12350,310,1,334.40


In [17]:
print(rfm.shape)
rfm.describe()

(5878, 4)


,Customer ID,Recency,Frequency,Monetary
count,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,2955.904095
std,1715.572666,209.338707,13.009406,14440.852688
min,12346.000000,1.000000,1.000000,2.950000
25%,13833.250000,26.000000,1.000000,342.280000
50%,15314.500000,96.000000,3.000000,867.740000
75%,16797.750000,380.000000,7.000000,2248.305000
max,18287.000000,739.000000,398.000000,580987.040000


In [18]:
rfm["R_Score"] = pd.qcut(rfm["Recency"], 4, labels=[4, 3, 2, 1]).astype(int)
rfm["F_Score"] = pd.qcut(rfm["Frequency"].rank(method="first"), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M_Score"] = pd.qcut(rfm["Monetary"], 4, labels=[1, 2, 3, 4]).astype(int)

rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score
0,12346,326,12,77556.46,2,4,4
1,12347,2,8,4921.53,4,4,4
2,12348,75,5,2019.40,3,3,3
3,12349,19,4,4428.69,4,3,4
4,12350,310,1,334.40,2,1,1


In [20]:
rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

In [21]:
def segment_customer(row):
    if row["RFM_Score"] >= 10:
        return "Champions"
    elif row["RFM_Score"] >= 8:
        return "Loyal Customers"
    elif row["RFM_Score"] >= 6:
        return "Potential Loyalist"
    elif row["RFM_Score"] >= 4:
        return "At Risk"
    else:
        return "Lost"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)
rfm.head()

,Customer ID,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score,Segment
0,12346,326,12,77556.46,2,4,4,10,Champions
1,12347,2,8,4921.53,4,4,4,12,Champions
2,12348,75,5,2019.40,3,3,3,9,Loyal Customers
3,12349,19,4,4428.69,4,3,4,11,Champions
4,12350,310,1,334.40,2,1,1,4,At Risk


In [22]:
rfm["Segment"].value_counts()

Segment
Champions             1740
Potential Loyalist    1216
Loyal Customers       1186
At Risk               1165
Lost                   571
Name: count, dtype: int64

In [23]:
rfm.to_csv("../data/rfm_segments.csv", index=False)

In [24]:
segment_summary = rfm.groupby("Segment").agg(
    Customers=("Customer ID", "count"),
    Avg_Recency=("Recency", "mean"),
    Avg_Frequency=("Frequency", "mean"),
    Avg_Monetary=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum")
).reset_index()

segment_summary["Revenue_Share_%"] = (segment_summary["Total_Revenue"] / segment_summary["Total_Revenue"].sum() * 100).round(1)
segment_summary["Customer_Share_%"] = (segment_summary["Customers"] / segment_summary["Customers"].sum() * 100).round(1)

segment_summary.sort_values("Total_Revenue", ascending=False)

,Segment,Customers,Avg_Recency,Avg_Frequency,Avg_Monetary,Total_Revenue,Revenue_Share_%,Customer_Share_%
1,Champions,1740,37.203448,15.221264,8056.420540,1.401817e+07,80.7,29.6
3,Loyal Customers,1186,116.580944,4.412310,1585.783427,1.880739e+06,10.8,20.2
4,Potential Loyalist,1216,213.304276,2.519737,796.007156,9.679447e+05,5.6,20.7
0,At Risk,1165,350.748498,1.387124,350.466447,4.082934e+05,2.3,19.8
2,Lost,571,547.162872,1.000000,174.527618,9.965527e+04,0.6,9.7
